In [151]:
import pandas as pd
import numpy as np
import re
pd.set_option('display.max_columns', None)

In [152]:
df = pd.read_csv("../../datasets/raw/steam_games_requirements.csv")

In [153]:
df.head(1)

,Unnamed: 0,url,types,name,desc_snippet,recent_reviews,all_reviews,release_date,developer,publisher,popular_tags,game_details,languages,achievements,genre,game_description,mature_content,minimum_requirements,recommended_requirements,original_price,discount_price
0,0,https://store.steampowered.com/app/379720/DOOM/,app,DOOM,Now includes all three premium DLC packs (Unto...,"Very Positive,(554),- 89% of the 554 user revi...","Very Positive,(42,550),- 92% of the 42,550 use...","May 12, 2016",id Software,"Bethesda Softworks,Bethesda Softworks","FPS,Gore,Action,Demons,Shooter,First-Person,Gr...","Single-player,Multi-player,Co-op,Steam Achieve...","English,French,Italian,German,Spanish - Spain,...",54.0,Action,"About This Game Developed by id software, the...",NaN,"Minimum:,OS:,Windows 7/8.1/10 (64-bit versions...","Recommended:,OS:,Windows 7/8.1/10 (64-bit vers...",$19.99,$14.99


In [154]:
df = df[df['types'] == 'app']
df.drop(columns=['types'], inplace=True)

In [155]:
df['url'] = df['url'].apply(lambda row: row.split('/')[4])

In [156]:
df.rename(columns={'url': 'app_id'}, inplace=True)

In [157]:
df = df[['app_id', 'name', 'minimum_requirements', 'recommended_requirements']]

In [158]:
df.head(1)

,app_id,name,minimum_requirements,recommended_requirements
0,379720,DOOM,"Minimum:,OS:,Windows 7/8.1/10 (64-bit versions...","Recommended:,OS:,Windows 7/8.1/10 (64-bit vers..."


In [159]:
df.isna().sum()

app_id                          0
name                           14
minimum_requirements        16952
recommended_requirements    16946
dtype: int64

In [206]:
def extract_cpu(strings):
    if pd.isna(strings):
        return None
    if 'Processor:' in strings:
        s = strings.split(',')
        return s[s.index('Processor:')+1]
    return None

In [194]:
req = df[df['minimum_requirements'].str.contains('Processor|CPU', case=True)]['minimum_requirements'].tolist()

In [237]:
for i in req:
    if 'CPU' in i and not 'Processor' in i:
        print(i, req.index(i))
        break

Minimum:,OS: 64-bit Windows 7 or later,CPU: Core i5-760 or better / AMD Phenom II X4 or better [Quad-core CPU],Memory: 6 GB RAM (64-bit),Hard Drive: 20 GB free,Video Card: nVidia GeForce GTX 260 or better / Radeon HD 4850 or better,DirectX®: 11.0 110


In [242]:
def extract_cpu2(text):
    """Extract CPU info from Processor line."""
    CPU_LINE_PATTERN = r'(?i)(?:Processor|CPU):[,\s]+([^,]+)'
    if pd.isna(text):
        return None
    text = str(text)
    match = re.search(CPU_LINE_PATTERN, text)
    if match:
        return match.group(1).strip()
    return None

In [247]:
extract_cpu2(req[10])

'Intel Core i3-4170 @ 3.70GHz'

In [239]:
req[110]

'Minimum:,OS: 64-bit Windows 7 or later,CPU: Core i5-760 or better / AMD Phenom II X4 or better [Quad-core CPU],Memory: 6 GB RAM (64-bit),Hard Drive: 20 GB free,Video Card: nVidia GeForce GTX 260 or better / Radeon HD 4850 or better,DirectX®: 11.0'